# Modelos Supervisionados — Projeto Lupa (Módulo 6)

Prevendo `houve_glosa`: Regressão Logística, KNN, Naive Bayes, SVM — mesmo split out-of-time do Módulo 5.

In [1]:
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)

import sys
sys.path.append("..")
from src.preprocessing import (
    carregar_dados, construir_pipeline_preprocessamento, criar_target,
    get_engine, split_out_of_time,
)

engine = get_engine()
df = criar_target(carregar_dados(engine))
treino, teste = split_out_of_time(df, ano_corte=2026, mes_corte=4)
print(f"Treino: {len(treino)} | Teste: {len(teste)}")

Treino: 585372 | Teste: 55146


## Decisão: undersampling no treino

KNN e SVM não escalam bem para 585 mil linhas (distância par-a-par fica caro demais). Decisão: aplicar `RandomUnderSampler` **só no treino** (nunca no teste - o teste tem que refletir a proporção real do mundo), reaproveitando a comparação já feita no Módulo 5. Isso deixa o treino com ~56 mil linhas balanceadas, viável para todos os 4 modelos.

In [2]:
pipeline_prep = construir_pipeline_preprocessamento()

X_treino = pipeline_prep.fit_transform(treino)  # fit só no treino - sem leakage
X_teste = pipeline_prep.transform(teste)         # teste só transforma, nunca re-ajusta

y_treino = treino["houve_glosa"].values
y_teste = teste["houve_glosa"].values

X_treino_bal, y_treino_bal = RandomUnderSampler(random_state=42).fit_resample(X_treino, y_treino)
print(f"Treino balanceado: {X_treino_bal.shape[0]} linhas")
print(f"Teste (intacto, proporção real): {X_teste.shape[0]} linhas, {y_teste.mean()*100:.2f}% com glosa")

# GaussianNB exige array denso
X_treino_bal_denso = X_treino_bal.toarray() if hasattr(X_treino_bal, "toarray") else X_treino_bal
X_teste_denso = X_teste.toarray() if hasattr(X_teste, "toarray") else X_teste

Treino balanceado: 56400 linhas
Teste (intacto, proporção real): 55146 linhas, 4.51% com glosa


## Treinando os 4 modelos

Decisão: `LinearSVC` em vez de `SVC(kernel="rbf")` — o SVM com kernel RBF tem custo computacional que cresce quadraticamente com o número de linhas, inviável em tempo razoável mesmo com 56 mil linhas. `LinearSVC` ainda é um SVM (kernel linear), só mais escalável. Como ele não gera probabilidade, usamos `decision_function` como score para as métricas baseadas em ranking (AUC-ROC, PR-AUC, KS).

In [3]:
from sklearn.svm import LinearSVC

modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Naive Bayes": GaussianNB(),
    "SVM (linear)": LinearSVC(max_iter=5000, random_state=42),
}

scores_teste = {}
predicoes_teste = {}

for nome, modelo in modelos.items():
    if nome in ("Naive Bayes",):
        modelo.fit(X_treino_bal_denso, y_treino_bal)
        pred = modelo.predict(X_teste_denso)
        score = modelo.predict_proba(X_teste_denso)[:, 1]
    else:
        modelo.fit(X_treino_bal, y_treino_bal)
        pred = modelo.predict(X_teste)
        score = (
            modelo.predict_proba(X_teste)[:, 1]
            if hasattr(modelo, "predict_proba")
            else modelo.decision_function(X_teste)
        )
    predicoes_teste[nome] = pred
    scores_teste[nome] = score
    print(f"{nome}: treinado")

Regressão Logística: treinado


KNN: treinado


Naive Bayes: treinado


SVM (linear): treinado


## Métricas — PR-AUC priorizada pelo desbalanceamento

Acurácia é enganosa aqui: um modelo que sempre prevê "sem glosa" já acertaria ~95,5% (a proporção da classe majoritária no teste), sem aprender nada de útil. PR-AUC (precision-recall) é mais informativa que AUC-ROC quando a classe positiva é rara, porque foca em quão bem o modelo distingue a classe minoritária, não penalizada pela enorme quantidade de negativos verdadeiros triviais.

In [4]:
def calcular_ks(y_real, score):
    fpr, tpr, _ = roc_curve(y_real, score)
    return np.max(tpr - fpr)

linhas_metricas = []
for nome in modelos:
    pred = predicoes_teste[nome]
    score = scores_teste[nome]
    linhas_metricas.append({
        "modelo": nome,
        "precision": precision_score(y_teste, pred),
        "recall": recall_score(y_teste, pred),
        "f1": f1_score(y_teste, pred),
        "auc_roc": roc_auc_score(y_teste, score),
        "pr_auc": average_precision_score(y_teste, score),
        "ks": calcular_ks(y_teste, score),
    })

tabela_metricas = pd.DataFrame(linhas_metricas).set_index("modelo").sort_values("pr_auc", ascending=False)
tabela_metricas.round(4)

,precision,recall,f1,auc_roc,pr_auc,ks
modelo,,,,,,
KNN,0.1810,0.7885,0.2944,0.8798,0.3205,0.6200
Regressão Logística,0.1596,0.7696,0.2644,0.8407,0.2624,0.5896
SVM (linear),0.1571,0.7720,0.2611,0.8402,0.2590,0.5897
Naive Bayes,0.0583,0.9513,0.1099,0.6583,0.0654,0.2982


## Tabela de custo: falso positivo vs falso negativo

Falso positivo = sinalizar uma despesa legítima para checagem manual (custo: tempo de um analista revisando algo que estava certo). Falso negativo = deixar passar uma despesa que teve glosa de verdade (custo: uma despesa irregular não é revisada). Assumimos, como estimativa de discussão de negócio, que o custo de investigar à toa é bem menor que o custo de deixar passar despesa com glosa real — refletindo a natureza pública/fiscalizatória do produto.

In [5]:
CUSTO_FALSO_POSITIVO = 50    # tempo de analista revisando despesa legítima
CUSTO_FALSO_NEGATIVO = 500   # despesa com glosa real não revisada

linhas_custo = []
for nome in modelos:
    pred = predicoes_teste[nome]
    tn, fp, fn, tp = confusion_matrix(y_teste, pred).ravel()
    custo_total = fp * CUSTO_FALSO_POSITIVO + fn * CUSTO_FALSO_NEGATIVO
    linhas_custo.append({
        "modelo": nome, "FP": fp, "FN": fn,
        "custo_fp": fp * CUSTO_FALSO_POSITIVO,
        "custo_fn": fn * CUSTO_FALSO_NEGATIVO,
        "custo_total": custo_total,
    })

tabela_custo = pd.DataFrame(linhas_custo).set_index("modelo").sort_values("custo_total")
tabela_custo

,FP,FN,custo_fp,custo_fn,custo_total
modelo,,,,,
KNN,8875,526,443750,263000,706750
Regressão Logística,10079,573,503950,286500,790450
SVM (linear),10298,567,514900,283500,798400
Naive Bayes,38205,121,1910250,60500,1970750


## Interpretando a Regressão Logística em odds ratio

Coeficiente da logística, sozinho, está em escala log-odds - difícil de interpretar. `exp(coeficiente)` vira **odds ratio**: quanto a chance de glosa multiplica quando aquela categoria/feature está presente, tudo mais constante.

In [6]:
nomes_features = pipeline_prep.get_feature_names_out()
modelo_logistica = modelos["Regressão Logística"]

odds_ratio = pd.DataFrame({
    "feature": nomes_features,
    "coeficiente": modelo_logistica.coef_[0],
    "odds_ratio": np.exp(modelo_logistica.coef_[0]),
}).sort_values("odds_ratio", ascending=False)

print("Top 10 - maior chance de glosa:")
print(odds_ratio.head(10).to_string(index=False))
print("\nTop 10 - menor chance de glosa:")
print(odds_ratio.tail(10).to_string(index=False))

Top 10 - maior chance de glosa:
                                                                 feature  coeficiente  odds_ratio
                                                cat__categoria_TELEFONIA     3.247295   25.720668
cat__categoria_MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE PARLAMENTAR     2.539346   12.671385
               cat__categoria_FORNECIMENTO DE ALIMENTAÇÃO DO PARLAMENTAR     2.489158   12.051125
            cat__categoria_LOCAÇÃO OU FRETAMENTO DE VEÍCULOS AUTOMOTORES     1.706898    5.511837
                cat__categoria_SERVIÇO DE TÁXI, PEDÁGIO E ESTACIONAMENTO     1.278978    3.592966
                                                       cat__partido_REDE     0.588220    1.800781
                               cat__categoria_PASSAGEM AÉREA - REEMBOLSO     0.569079    1.766639
        cat__categoria_PARTICIPAÇÃO EM CURSO, PALESTRA OU EVENTO SIMILAR     0.508729    1.663176
              cat__categoria_PASSAGENS TERRESTRES, MARÍTIMAS OU FLUVIAIS     0.360118 